# `metrics.py` 

---

## Functions

| Function | Returns | Inputs | Opportunity-corrected? |
|---|---|---|---|
| `compute_spc` | P(recall) per serial position | `recall_sims, N` | N/A |
| `compute_pfr` | P(first recall) per serial position | `recall_sims, N` | N/A |
| `recall_accuracy` | Mean fraction of unique items recalled | `recall_sims, N` | N/A |
| `compute_lag_crp` | CRP curve (includes lag 0) | `recall_sims, N` | **Yes** |
| `lag_crp_with_counts` | CRP + num + den arrays (excludes lag 0) | `recall_sims, N` | **Yes** |
| `conditional_forward_lag_rates` | P(l=+1), P(l>=k) | `lags, num, den` | **Yes** |
| `conditional_backward_lag_rates` | P(l=-1), P(l<=-k) | `lags, num, den` | **Yes** |
| `conditional_abs_lag_summaries` | P(\|l\|=1), P(\|l\|>=k), E[\|l\|] | `lags, num, den` | **Yes** |
| `unconditional_transition_summaries` | P(\|l\|=1), P(\|l\|>=k), E[\|l\|] | `recall_sims` | **No** |

---

## Primary recall metrics

### `compute_spc`

For each serial position $j \in \{1,\dots,N\}$:

$$\text{SPC}(j) = \frac{1}{S}\sum_{s=1}^{S} \mathbf{1}\bigl[j \in \text{recalls}_s\bigr]$$

where $S$ is the number of trials. The indicator is 1 if serial position $j$ appears *anywhere* in trial $s$'s recall sequence.

#### Algorithm:
loops `j` from 1 to $N$, uses `np.any(recall_sims == j, axis=0)` to get a boolean per trial, then `np.mean` gives the probability.

### `compute_pfr`

$$\text{PFR}(j) = \frac{\#\{\text{trials where first recall} = j\}}{\#\{\text{trials with at least one recall}\}}$$

Extracts `recall_sims[0, :]` (the first output position), filters out zeros (no-recall trials), then counts how often each position $j$ appears.

### `recall_accuracy`

$$\text{Accuracy} = \frac{1}{S}\sum_{s=1}^{S}\frac{|\{\text{unique items recalled in trial } s\}|}{N}$$

#### Algorithm:
per trial, extracts non-zero entries, optionally takes `np.unique`, divides count by $N$, averages across trials.



### `lag_crp_with_counts`

At each transition $t \to t+1$ in a trial, the subject has already recalled some items. Only the *remaining* items are available for the next transition. **Opportunity correction** asks: "given which lags were *possible*, how often was each lag *chosen*?"

For each transition from current position $i$ to next position $j$:

$$ 
\text{lag} = j - i 
$$

- `num[l]` counts: "how many times was lag $l$ actually chosen?"
- `den[l]` counts: "how many times was lag $l$ available (i.e., $i + l$ was a remaining item)?"
- `CRP(l)` = `num[l]` / `den[l]`

#### Algorithm:
```text

for each trial s:
    for each transition t -> t+1:
        cur = recall[t], nxt = recall[t+1]
        add cur to recalled set
        remaining = {1..N} \ recalled
        for each possible lag l:
            if (cur + l) is in remaining:
                den[l] += 1          # this lag was available
        num[nxt - cur] += 1          # this lag was chosen

```

Key detail: `cur` is added to `recalled` *before* computing opportunities, so it cannot be an opportunity for itself. The function excludes lag 0.

---

## Conditional directional lag summaries

These functions take the `lags`, `num`, `den` arrays from `lag_crp_with_counts` and produce scalar summaries for *signed* lags.

### Forward: `conditional_forward_lag_rates`

| Metric | Formula | Code (lines 117–121) |
|---|---|---|
| $P(\ell = +1)$ | $\displaystyle\frac{\text{num}(+1)}{\text{den}(+1)}$ | `m = valid & (lags == 1)` then `num[m].sum() / den[m].sum()` |
| $P(\ell \geq k)$ | $\displaystyle\frac{\sum_{\ell \geq k}\text{num}(\ell)}{\sum_{\ell \geq k}\text{den}(\ell)}$ | `m = valid & (lags >= large_lag_thresh)` then same ratio |

### Backward: `conditional_backward_lag_rates`

| Metric | Formula | Code (lines 136–140) |
|---|---|---|
| $P(\ell = -1)$ | $\displaystyle\frac{\text{num}(-1)}{\text{den}(-1)}$ | `m = valid & (lags == -1)` then `num[m].sum() / den[m].sum()` |
| $P(\ell \leq -k)$ | $\displaystyle\frac{\sum_{\ell \leq -k}\text{num}(\ell)}{\sum_{\ell \leq -k}\text{den}(\ell)}$ | `m = valid & (lags <= -large_lag_thresh)` then same ratio |

Both use the **pool-then-divide** pattern: sum all relevant numerators, sum all relevant denominators, single division.

---

## Unconditional transition summaries 
### `unconditional_transition_summaries`

**Code (lines 153–167):** `get_lags_from_recall_sims` pools all `np.diff(seq)` across trials. Then:

$$P(|\ell| = 1) = \frac{\#\{|\ell_i| = 1\}}{\#\{\text{all lags}\}}, \qquad
P(|\ell| \geq k) = \frac{\#\{|\ell_i| \geq k\}}{\#\{\text{all lags}\}}$$

The denominator is always the total number of transitions; it does not account for which lags were *available*. If a parameter setting causes fewer items to be recalled (fewer transitions), the distribution of possible lags changes mechanically, which can bias these summaries.

---

## Conditional absolute-value lag summaries
### `conditional_abs_lag_summaries`

This is the opportunity-corrected counterpart of `unconditional_transition_summaries`. It answers: **ignoring direction, how contiguous are transitions after correcting for what was available?**

For each absolute distance $d \in \{1, \dots, N-1\}$, define:

$$\text{num}_{\text{abs}}(d) = \text{num}(+d) + \text{num}(-d), \qquad
\text{den}_{\text{abs}}(d) = \text{den}(+d) + \text{den}(-d)$$

This collapses the signed lag-CRP arrays into absolute-distance buckets.

$$\text{metric}(S) = \frac{\sum_{d \in S}\text{num}_{\text{abs}}(d)}{\sum_{d \in S}\text{den}_{\text{abs}}(d)}$$

| Metric | Subset $S$ | Formula | Code reference |
|---|---|---|---|
| $P(\lvert\ell\rvert = 1 \mid \text{opp})$ | $\{d : \lvert\ell\rvert = 1\}$ | $\dfrac{\text{num}_{\text{abs}}(1)}{\text{den}_{\text{abs}}(1)}$ | lines 247–248 |
| $P(\lvert\ell\rvert \geq k \mid \text{opp})$ | $\{d : \lvert\ell\rvert \geq k\}$ | $\dfrac{\sum_{d \geq k}\text{num}_{\text{abs}}(d)}{\sum_{d \geq k}\text{den}_{\text{abs}}(d)}$ | lines 251–252 |

---

## Two lag-CRP functions

### `compute_lag_crp` vs `lag_crp_with_counts`

Both compute the same opportunity-corrected CRP. The differences:

| | `compute_lag_crp` | `lag_crp_with_counts` |
|---|---|---|
| **Includes lag 0?** | Yes (always CRP=0 at lag 0) | No |
| **Returns num/den?** | No (only `lag_vals, crp`) | Yes (`lags, crp, num, den`) |
| **Use case** | Plotting the CRP curve | Input to all scalar summary functions |

`lag_crp_with_counts` is the workhorse: its `num` and `den` arrays are the inputs to every conditional summary function.